# APS 8 — Coverage Path Planning com Frontier Memory + Curriculum Learning

**Disciplina:** Reinforcement Learning — Insper 2026  
**Autor:** Luigi Zema Matizonkas

Este notebook foi desenvolvido em **Jupyter** (não em repositório clonado), o que facilita iteração rápida e visualização inline das curvas de treino. Todo o desenvolvimento — desde os primeiros experimentos até os resultados finais — foi feito aqui.

## Resultados

| Ambiente | Meta | Resultado | Status |
|---|---|---|---|
| 5×5  | ≥ 90/100 cobertura completa | **97/100** | ✅ Ponto 1 |
| 10×10 | ≥ 90/100 cobertura completa | **91/100** | ✅ Ponto 2 |
| 20×20 (bônus) | ≥ 90/100 cobertura completa | **78/100** | ❌ plateau arquitetural |

**Abordagem:** `FrontierMemoryWrapper` — wrapper Gymnasium que mantém mapa interno acumulado e expõe patch 5×5 invariante ao tamanho do grid + vetor de 26 features escalares. PPO com Curriculum Learning 5×5 → 10×10.

---
## Seção 1 — Instalação

In [ ]:
import subprocess, sys
for pkg in ["gymnasium>=1.0","stable-baselines3>=2.0",
            "torch>=2.0","numpy","matplotlib","pandas","seaborn","pygame"]:
    r = subprocess.run([sys.executable,"-m","pip","install","-q",pkg],
                       capture_output=True,text=True)
    print(f"[{'OK' if r.returncode==0 else 'ERRO'}] {pkg}")

---
## Seção 2 — Imports e Configuração

In [ ]:
import os, sys, io, json, time, zipfile, warnings
from pathlib import Path

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

# APS8_DIR aponta para a pasta onde este notebook está salvo.
# Em Jupyter, Path.cwd() retorna o diretório do notebook quando aberto normalmente.
APS8_DIR = Path.cwd()
os.chdir(APS8_DIR)
sys.path.insert(0, str(APS8_DIR))

DEVICE = "cpu"
SEED   = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch  : {torch.__version__}")
print(f"SB3      : {__import__('stable_baselines3').__version__}")
print(f"Gym      : {gym.__version__}")
print(f"Dir      : {APS8_DIR}")
print(f"  ppo_88_5x5.zip   : {'✅' if (APS8_DIR/'ppo_88_5x5.zip').exists() else '❌ não encontrado'}")
print(f"  ppo_88_10x10.zip : {'✅' if (APS8_DIR/'ppo_88_10x10.zip').exists() else '❌ não encontrado'}")
print(f"  grid_world_cpp.py: {'✅' if (APS8_DIR/'grid_world_cpp.py').exists() else '❌ não encontrado'}")

---
## Seção 3 — Ambiente GridWorldCPPEnv

In [ ]:
from grid_world_cpp import GridWorldCPPEnv

env_demo = GridWorldCPPEnv(size=5, obs_quantity=3, max_steps=200)
obs, info = env_demo.reset(seed=SEED)
print(f"agent shape    : {obs['agent'].shape}")
print(f"neighbors shape: {obs['neighbors'].shape}")
print(f"Células livres : {env_demo.total_free_cells}")
env_demo.close()

---
## Seção 4 — `FrontierMemoryWrapper`

Mantém mapa interno acumulado (`_mem`) e extrai sinais escalares + patch visual 5×5:

### Vetor `agent` (26 valores)
| Campo | Descrição |
|---|---|
| `x_norm`, `y_norm` | Posição normalizada [0,1] |
| `coverage` | Fração de células visitadas |
| `dx_front`, `dy_front` | Direção ao frontier mais próximo ([-1,1]) |
| `dist_front` | Distância Manhattan ao frontier (normalizada) |
| `cnt_N/E/S/W` | Fração de não-visitados por quadrante |
| `act_t-3..t` (×16) | One-hot das 4 últimas ações — evita loops |

### Patch `neighbors` (5×5 = 25 valores)
Extrato do `_mem` centrado no agente. **Invariante ao tamanho do grid** — mesmo formato em 5×5, 10×10 ou 20×20.

| Valor | Significado |
|---|---|
| 0.00 | Desconhecido (ainda não visto via vizinhança) |
| 0.33 | Obstáculo ou fora dos limites do grid |
| 0.67 | Visitado pelo agente |
| 1.00 | Livre mas ainda não visitado |

### Reward shaping no wrapper
| Evento | Reward env | Reward total |
|---|---|---|
| Cobertura completa | +10 | **+25** (+15 extra) |
| Truncamento (sem cobertura) | -5 | **-3** (+2 suavizador) |

In [ ]:
class FrontierMemoryWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self._mem = None
        self._action_hist = np.zeros(16, dtype=np.float32)
        low_extra  = np.zeros(16, dtype=np.float32)
        high_extra = np.ones(16,  dtype=np.float32)
        self.observation_space = gym.spaces.Dict({
            "agent": gym.spaces.Box(
                low =np.concatenate([[ 0., 0., 0., -1.,-1., 0.,  0., 0., 0., 0.], low_extra ]).astype(np.float32),
                high=np.concatenate([[ 1., 1., 1.,  1., 1., 1.,  1., 1., 1., 1.], high_extra]).astype(np.float32),
            ),
            "neighbors": gym.spaces.Box(low=0.0, high=1.0, shape=(5, 5), dtype=np.float32),
        })

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        sz = self.env.size
        self._mem = np.zeros((sz, sz), dtype=np.float32)
        self._action_hist = np.zeros(16, dtype=np.float32)
        self._update_mem()
        return self._build_obs(obs), info

    def step(self, action):
        one_hot = np.zeros(4, dtype=np.float32)
        one_hot[int(action)] = 1.0
        self._action_hist = np.roll(self._action_hist, -4)
        self._action_hist[-4:] = one_hot
        obs, r, term, trunc, info = self.env.step(action)
        self._update_mem()
        if term and info.get("coverage", 0) >= 0.999:
            r += 15.0
        if trunc:
            r += 2.0
        return self._build_obs(obs), r, term, trunc, info

    def _update_mem(self):
        ax = int(self.env._agent_location[0])
        ay = int(self.env._agent_location[1])
        nb = self.env._neighbors
        sz = self.env.size
        for i in range(3):
            for j in range(3):
                nx, ny = ax+(j-1), ay+(i-1)
                if 0 <= nx < sz and 0 <= ny < sz:
                    v = nb[i, j]
                    if v == 1:
                        self._mem[ny, nx] = 1.0
                    elif v == 2:
                        self._mem[ny, nx] = 2.0
                    elif v == 0 and self._mem[ny, nx] == 0.0:
                        self._mem[ny, nx] = 3.0
        self._mem[ay, ax] = 2.0

    def _get_mem_patch(self):
        ax = int(self.env._agent_location[0])
        ay = int(self.env._agent_location[1])
        sz = self.env.size
        patch = np.full((5, 5), 1.0/3.0, dtype=np.float32)
        for i in range(5):
            for j in range(5):
                nx = ax + (j - 2)
                ny = ay + (i - 2)
                if 0 <= nx < sz and 0 <= ny < sz:
                    patch[i, j] = self._mem[ny, nx] / 3.0
        return patch

    def _frontier_signal(self):
        ax = int(self.env._agent_location[0])
        ay = int(self.env._agent_location[1])
        sz = self.env.size
        total = max(self.env.total_free_cells, 1)
        unvis = (self._mem == 0) | (self._mem == 3)
        rows, cols = np.where(unvis)
        if len(rows) == 0:
            dx, dy, dist = 0.0, 0.0, 0.0
        else:
            d = np.abs(rows - ay) + np.abs(cols - ax)
            idx = int(np.argmin(d))
            ty, tx = rows[idx], cols[idx]
            dx   = float(tx - ax) / max(sz - 1, 1)
            dy   = float(ty - ay) / max(sz - 1, 1)
            dist = float(d[idx]) / max(2*(sz-1), 1)
        cnt_N = float(np.sum(unvis[:ay, :])) / total
        cnt_S = float(np.sum(unvis[ay+1:, :])) / total
        cnt_W = float(np.sum(unvis[:, :ax])) / total
        cnt_E = float(np.sum(unvis[:, ax+1:])) / total
        return dx, dy, dist, cnt_N, cnt_E, cnt_S, cnt_W

    def _build_obs(self, env_obs):
        dx, dy, dist, cN, cE, cS, cW = self._frontier_signal()
        base = env_obs["agent"]
        ext  = np.array([base[0], base[1], base[2], dx, dy, dist, cN, cE, cS, cW], dtype=np.float32)
        return {"agent": np.concatenate([ext, self._action_hist]), "neighbors": self._get_mem_patch()}

print("FrontierMemoryWrapper OK (patch 5×5 + frontier signals + action hist + reward shaping)")

In [ ]:
env_t = FrontierMemoryWrapper(GridWorldCPPEnv(size=5, obs_quantity=2, max_steps=200))
obs_t, _ = env_t.reset(seed=7)
for _ in range(8):
    obs_t, _, term, trunc, _ = env_t.step(env_t.action_space.sample())
    if term or trunc: break
print(f"agent    : {obs_t['agent'].shape}  (esperado: (26,))")
print(f"neighbors: {obs_t['neighbors'].shape}  (esperado: (5, 5))")
env_t.close()

---
## Seção 5 — Política MLP (PPO)

`MultiInputPolicy` do `stable_baselines3`: MLP 256 → 256 → 128.

```
agent  (26) ──┐
               ├── Dense 256 → Dense 256 → Dense 128 → π(a|s)  [actor]
patch  (5×5) ──┘                                      → V(s)   [critic]
```

> `normalize_images=False` — patch 5×5 tratado como vetor escalar pelo SB3.
> `deterministic=False` na avaliação — estocasticidade quebra loops.

In [ ]:
POLICY_KWARGS = dict(
    net_arch=dict(pi=[256, 256, 128], vf=[256, 256, 128]),
    normalize_images=False,
)
print("Policy kwargs definidos (MLP 256-256-128).")

---
## Seção 6 — Validação Rápida

In [ ]:
def _validate():
    venv = DummyVecEnv([lambda: FrontierMemoryWrapper(
        GridWorldCPPEnv(size=5, obs_quantity=3, max_steps=50))])
    m = PPO("MultiInputPolicy", venv, policy_kwargs=POLICY_KWARGS,
            verbose=0, device=DEVICE, n_steps=64, batch_size=32)
    m.learn(total_timesteps=128)
    venv.close()
    print(f"[OK] Arquitetura MLP validada em '{DEVICE}'. Pode treinar.")
_validate()

---
## Seção 7 — Funções Auxiliares

In [ ]:
class TrainingCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.ep_rewards, self.ep_timesteps = [], []
    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.ep_rewards.append(info["episode"]["r"])
                self.ep_timesteps.append(self.num_timesteps)
        return True

def make_venv(size, obstacles, max_steps, n_envs=8):
    def _init():
        return Monitor(FrontierMemoryWrapper(
            GridWorldCPPEnv(size=size, obs_quantity=obstacles, max_steps=max_steps)))
    return DummyVecEnv([_init] * n_envs)

def bypass_load(zip_path, env):
    p = str(zip_path)
    if not p.endswith(".zip"): p += ".zip"
    with zipfile.ZipFile(p, "r") as z:
        with z.open("policy.pth") as f:
            buf = io.BytesIO(f.read())
    try:
        sd = torch.load(buf, map_location=DEVICE, weights_only=True)
    except Exception:
        buf.seek(0)
        sd = torch.load(buf, map_location=DEVICE, weights_only=False)
    m = PPO("MultiInputPolicy", env, policy_kwargs=POLICY_KWARGS, device=DEVICE, verbose=0)
    miss, unex = m.policy.load_state_dict(sd, strict=False)
    print(f"  bypass OK: missing={len(miss)}, unexpected={len(unex)}")
    return m

def evaluate_model(model, size, obstacles, max_steps, n_episodes=100, seed=0):
    results = []
    for ep in range(n_episodes):
        e = FrontierMemoryWrapper(GridWorldCPPEnv(size=size, obs_quantity=obstacles, max_steps=max_steps))
        obs, _ = e.reset(seed=seed+ep)
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=False)
            obs, _, term, trunc, info = e.step(int(action))
            done = term or trunc
        results.append({"coverage": info["coverage"]*100,
                         "steps": info["steps"],
                         "full": info["coverage"] >= 0.999})
        e.close()
    return pd.DataFrame(results)

def print_results(df, label):
    full = int(df["full"].sum())
    print(f"\n{'='*50}\n  {label}\n{'='*50}")
    print(f"  Cobertura completa : {full}/100  {'✅' if full >= 90 else '❌'}")
    print(f"  Cobertura média    : {df['coverage'].mean():.1f}% ± {df['coverage'].std():.1f}%")
    print(f"  Passos médios      : {df['steps'].mean():.0f}")

def smooth(v, w=200):
    return pd.Series(v).rolling(w, min_periods=1).mean().values

def plot_curve(cb, title, color="steelblue", save_path=None):
    if not cb.ep_rewards: return
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(cb.ep_timesteps, cb.ep_rewards, alpha=0.15, color=color, lw=0.5)
    ax.plot(cb.ep_timesteps, smooth(cb.ep_rewards, 200), color=color, lw=2, label="Média móvel")
    ax.set_xlabel("Timesteps"); ax.set_ylabel("Reward"); ax.set_title(title); ax.legend()
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()

print("Helpers OK")

---
## Seção 8 — Configuração dos Estágios

In [ ]:
N_ENVS = 8

DIM1, OBS1, STEPS1, TOTAL_TS_1 = 5,  3,  200,    600_000
DIM2, OBS2, STEPS2, TOTAL_TS_2 = 10, 12, 500,  1_500_000

MODEL_PATH_1 = APS8_DIR / "ppo_88_5x5"
MODEL_PATH_2 = APS8_DIR / "ppo_88_10x10"

SKIP_IF_EXISTS = True

print(f"E1: {DIM1}x{DIM1}  | {TOTAL_TS_1:,} steps | max_steps={STEPS1}")
print(f"E2: {DIM2}x{DIM2} | {TOTAL_TS_2:,} steps | max_steps={STEPS2}")
print(f"SKIP_IF_EXISTS={SKIP_IF_EXISTS}")
print(f"  ppo_88_5x5.zip   : {'existe' if MODEL_PATH_1.with_suffix('.zip').exists() else 'não encontrado'}")
print(f"  ppo_88_10x10.zip : {'existe' if MODEL_PATH_2.with_suffix('.zip').exists() else 'não encontrado'}")

---
## Seção 9 — Treinamento Estágio 1: 5×5

In [ ]:
cb_1 = TrainingCallback()
elapsed_1 = 0.0

if SKIP_IF_EXISTS and MODEL_PATH_1.with_suffix(".zip").exists():
    print(f"SKIP E1: carregando {MODEL_PATH_1.name}.zip")
    _d = make_venv(DIM1, OBS1, STEPS1, n_envs=N_ENVS)
    model_1 = bypass_load(MODEL_PATH_1, _d)
    _d.close()
else:
    venv_1 = make_venv(DIM1, OBS1, STEPS1, n_envs=N_ENVS)
    model_1 = PPO(
        "MultiInputPolicy", venv_1,
        policy_kwargs=POLICY_KWARGS,
        ent_coef=0.03, learning_rate=3e-4,
        n_steps=2048, batch_size=256,
        gae_lambda=0.95, gamma=0.99,
        verbose=0, device=DEVICE, seed=SEED,
    )
    t0 = time.time()
    model_1.learn(total_timesteps=TOTAL_TS_1, callback=cb_1, progress_bar=True)
    elapsed_1 = time.time() - t0
    model_1.save(str(MODEL_PATH_1))
    venv_1.close()

print(f"E1 pronto em {elapsed_1/60:.1f} min")

In [ ]:
plot_curve(cb_1, "Curva — E1: Frontier MLP (5×5)", "steelblue", APS8_DIR / "curve_frontier_5x5.png")

---
## Seção 10 — Avaliação Estágio 1

In [ ]:
print("Avaliando E1...")
df1_5x5   = evaluate_model(model_1, size=5,  obstacles=3,  max_steps=200)
print_results(df1_5x5,   "E1 → 5×5")
df1_10x10 = evaluate_model(model_1, size=10, obstacles=12, max_steps=500)
print_results(df1_10x10, "E1 → 10×10 (zero-shot)")

---
## Seção 11 — Curriculum Estágio 2: 10×10

In [ ]:
cb_2 = TrainingCallback()
elapsed_2 = 0.0

if SKIP_IF_EXISTS and MODEL_PATH_2.with_suffix(".zip").exists():
    print(f"SKIP E2: carregando {MODEL_PATH_2.name}.zip")
    _d = make_venv(DIM2, OBS2, STEPS2, n_envs=N_ENVS)
    model_2 = bypass_load(MODEL_PATH_2, _d)
    _d.close()
else:
    venv_2 = make_venv(DIM2, OBS2, STEPS2, n_envs=N_ENVS)
    model_1.set_env(venv_2)
    for pg in model_1.policy.optimizer.param_groups: pg["lr"] = 1e-4
    model_1.learning_rate = 1e-4
    model_1.ent_coef = 0.03
    t0 = time.time()
    model_1.learn(total_timesteps=TOTAL_TS_2, callback=cb_2,
                  reset_num_timesteps=True, progress_bar=True)
    elapsed_2 = time.time() - t0
    model_2 = model_1
    model_2.save(str(MODEL_PATH_2))
    venv_2.close()

print(f"E2 pronto em {elapsed_2/60:.1f} min")

In [ ]:
plot_curve(cb_2, "Curva — E2: Curriculum (10×10)", "darkorange", APS8_DIR / "curve_frontier_10x10.png")

---
## Seção 12 — Avaliação Estágio 2

In [ ]:
print("Avaliando E2...")
df2_10x10 = evaluate_model(model_2, size=10, obstacles=12, max_steps=500)
print_results(df2_10x10, "E2 → 10×10")
df2_20x20 = evaluate_model(model_2, size=20, obstacles=48, max_steps=1000)
print_results(df2_20x20, "E2 → 20×20 zero-shot (max_steps=1000)")

---
## Seção 15 — Comparação Final

In [ ]:
rows = [
    ("Baseline (sem memória)",    "5×5",   75,  95.0),
    ("Baseline (sem memória)",    "10×10", 65,  85.0),
    ("8.7 Frontier 3×3 — E1",     "5×5",   87,  96.3),
    ("8.7 Frontier 3×3 — E1",     "10×10", 27,  72.0),
    ("8.8 Frontier 5×5 — E1",     "5×5",   int(df1_5x5["full"].sum()),   round(df1_5x5["coverage"].mean(),1)),
    ("8.8 Frontier 5×5 — E2",     "10×10", int(df2_10x10["full"].sum()), round(df2_10x10["coverage"].mean(),1)),
]
df_sum = pd.DataFrame(rows, columns=["Modelo","Ambiente","Cobertura completa (/100)","Cobertura média (%)"])
print(df_sum.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
configs = [
    ("5×5",  [75, 87, int(df1_5x5["full"].sum())],
             ["Baseline","8.7 (3×3)","8.8 E1 (5×5)"]),
    ("10×10",[65, 27, int(df2_10x10["full"].sum())],
             ["Baseline","8.7 zero-shot","8.8 E2"]),
]
colors = ["#d9534f","#aaaaaa","#5cb85c"]
for ax,(title,vals,labels) in zip(axes,configs):
    bars = ax.bar(labels, vals, color=colors[:len(vals)], edgecolor="white")
    ax.set_ylim(0, 115)
    ax.axhline(90, color="red", ls="--", lw=1.5, label="Meta 90%")
    ax.set_title(f"Ambiente {title}", fontweight="bold", fontsize=12)
    ax.set_ylabel("Cobertura completa /100")
    ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+2, str(v),
                ha="center", fontweight="bold", fontsize=11)
    ax.legend(fontsize=8)
plt.suptitle("APS 8 Final — Frontier Memory 5×5 Patch", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(APS8_DIR / "coverage_comparison_88.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for cb, label, color in [(cb_1,"E1 5×5","steelblue"), (cb_2,"E2 10×10","darkorange")]:
    if cb.ep_rewards:
        ax.plot(cb.ep_timesteps, smooth(cb.ep_rewards, 300), label=label, lw=2, color=color)
ax.set_xlabel("Timesteps"); ax.set_ylabel("Reward médio")
ax.set_title("Curvas de Treino — APS 8 Frontier Memory 5×5", fontsize=12)
ax.legend(); plt.tight_layout()
plt.savefig(APS8_DIR / "all_curves_final.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Seção 18 — Exportação JSON

In [ ]:
def to_m(df):
    return {"full_coverage_count": int(df["full"].sum()),
            "full_coverage_rate": round(float(df["full"].mean()*100), 1),
            "mean_coverage": round(float(df["coverage"].mean()), 2),
            "std_coverage":  round(float(df["coverage"].std()),  2),
            "mean_steps":    round(float(df["steps"].mean()), 1)}

results = {
    "E1_5x5":        to_m(df1_5x5),
    "E1_10x10_zero": to_m(df1_10x10),
    "E2_10x10":      to_m(df2_10x10),
    "E2_20x20_zero": to_m(df2_20x20),
    "status": {
        "ponto1": "GARANTIDO" if df1_5x5["full"].sum() >= 90 else "FALHOU",
        "ponto2": "GARANTIDO" if df2_10x10["full"].sum() >= 90 else "FALHOU",
    }
}

out = APS8_DIR / "evaluation_results_final.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"Salvo: {out.name}")
print(json.dumps(results, indent=2, ensure_ascii=False))

---
## Seção 19 — Conclusão

### ✅ Pontos garantidos: 2/2

| Objetivo | Resultado | Meta | Status |
|---|---|---|---|
| 5×5 cobertura completa | **97/100** | ≥ 90% | ✅ |
| 10×10 cobertura completa | **91/100** | ≥ 90% | ✅ |

### O que garantiu os resultados

1. **`FrontierMemoryWrapper`** — mapa interno 5×5 invariante ao tamanho do grid. O agente vê onde ainda não foi via patch normalizado centrado em si mesmo. Saída `_mem`: 0=desconhecido, 1=obstáculo, 2=visitado, 3=fronteira.

2. **Reward shaping** — cobertura completa: +25 total (vs +10 env); truncamento: -3 efetivo (vs -5 env).

3. **Curriculum Learning** — E1 (5×5, 600k steps) → E2 (10×10, 1.5M via `set_env()`). O modelo E1 serve de ponto de partida; o 10×10 aprende a generalizar em ~20 min.

4. **Generalização zero-shot** — E1 treinado só no 5×5 atinge **90/100 no 10×10** sem nenhum retreinamento. Valida a invariância do patch 5×5.

### Referências

- Repositório do professor: https://github.com/fbarth/gym_custom_env
- Aula 23: https://insper.github.io/rl/classes/23_custom_env_agent/
- Stable-Baselines3: https://stable-baselines3.readthedocs.io — Raffin et al. (2021)
- Schulman et al. (2017). *Proximal Policy Optimization Algorithms*. arXiv:1707.06347

**Autor:** Luigi Zema Matizonkas — Insper, Reinforcement Learning, 2026

---
## Seção 20 — Tentativa de Bônus: 20×20

Além dos 2 pontos obrigatórios, foram realizadas **10 tentativas distintas** de atingir o bônus (≥90/100 no 20×20 com `max_steps=1000`).

### Histórico completo de experimentos

| Tentativa | Estratégia | max_steps | 20×20 | Diagnóstico |
|---|---|---|---|---|
| 8.8 E3-v2 | Fine-tune direto | **4000 ⚠️** | 78/100 | Budget 4× acima do padrão |
| 8.9 LSTM | RecurrentPPO + LSTM | 4000 | **44/100** ❌ | Catastrophic forgetting |
| 8.9 MLP | Progressive reward + 15×15 | 4000 | 76/100 | Regressão no 10×10 |
| 8.12 V1 | Potential-Based Shaping (Ng 1999) | **5000 ⚠️** | 78/100 | mean_steps=1521 |
| 8.12 V2 | Potential + Revisit Neutralization | **5000 ⚠️** | 78/100 | mesmo plateau |
| 8.12 V3 | CNN feature extractor | **5000 ⚠️** | 73/100 | CNN pior que MLP |
| **8.13 V1** | Fine-tune, max_steps=1000 | **1000 ✅** | 75/100 | Budget correto; mean_steps=642 |
| **8.13 V2** | Retrain E2+E3, max_steps=1000 | **1000 ✅** | 76/100 | mean_steps=682 |
| **8.13 V3** | Currículo de obstáculos (16→32→48) | **1000 ✅** | **78/100** | **Melhor resultado** |
| **8.1 Deep-E3** | n_steps=4096, γ=0.998, near-miss | **1000 ✅** | 77/100 | Teto arquitetural confirmado |

### Diagnóstico de credit assignment

Com `n_steps=2048, N=8 envs` (configuração original), o PPO atualiza a cada 256 passos por ambiente. Um episódio 20×20 tem ~641 passos → ocupa 2.5 rollouts. O reward das últimas células chega com `γ^641 = 0.99^641 ≈ 0.001` — praticamente zero.

| Parâmetro | Configuração original | Deep-E3 (correção) |
|---|---|---|
| `n_steps` | 2048 | **4096** |
| Passos/env | 256 | **1024** |
| `gamma` | 0.99 | **0.998** |
| Desconto em 641 passos | 0.001 | **0.28** (280×) |
| Episódio por rollout | 2.5× (fragmentado) | **0.63×** (completo) |

### Melhor modelo 20×20

**8.13 V3** (78/100, max_steps=1000): currículo de obstáculos progressivo em 3 sub-estágios (16→32→48).  
Salvo em `ppo_best_20x20.pt`.

In [ ]:
def load_pt(pt_path, size, obs_q, max_steps):
    """Carrega modelo .pt (torch.save) — portátil, imune ao bug PyTorch >=2.1."""
    ck = torch.load(str(pt_path), map_location=DEVICE, weights_only=False)
    _d = make_venv(size, obs_q, max_steps, n_envs=1)
    m  = PPO("MultiInputPolicy", _d, policy_kwargs=POLICY_KWARGS, device=DEVICE, verbose=0)
    m.policy.load_state_dict(ck["policy_state_dict"])
    _d.close()
    return m

PT_20 = APS8_DIR / "ppo_best_20x20.pt"

if PT_20.exists():
    print(f"Carregando: {PT_20.name}")
    model_best20 = load_pt(PT_20, size=20, obs_q=48, max_steps=1000)
    print("Avaliando 100 episódios (max_steps=1000, padrão do professor)...")
    df_best20 = evaluate_model(model_best20, size=20, obstacles=48, max_steps=1000, n_episodes=100)
    print_results(df_best20, "Melhor modelo 20×20 — 8.13 V3 (max_steps=1000)")
    full20 = int(df_best20["full"].sum())
    status = "✅ BÔNUS ATINGIDO!" if full20 >= 90 else f"⚠️ {full20}/100 — próximo, bônus não atingido"
    print(f"\n  {status}")
    print(f"  Cobertura média : {df_best20['coverage'].mean():.2f}%")
    print(f"  Passos médios   : {df_best20['steps'].mean():.0f}")
else:
    print(f"AVISO: {PT_20.name} não encontrado. Coloque o arquivo na mesma pasta do notebook.")